
## Working with PySpark DataFrames for Analytics
**Day 42 | Batch 13 | 03 Mei 2026**

---

| Field | Isi |
|-------|-----|
| **Source** | Hive: `adventureworks.*` |
| **Target** | Hive: `adventureworks_curated.fact_sales_performance` |

---

## 🎯 Business Question

1. **Produk apa yang paling laris per wilayah?** (berdasarkan total revenue & total qty)
2. **Bagaimana tren revenue per bulan per tahun?** (YoY & MoM growth)
3. **Produk mana yang memiliki margin tertinggi?** (ListPrice - StandardCost)
4. **Top 5 produk per territory per tahun** (menggunakan Window Function)

## 📐 Pipeline yang Dibangun

```
Hive External Tables  →  Spark DataFrame  →  Transform  →  Curated Hive Table
fact_sales_orders         enrich + join        aggregate     fact_sales_performance
fact_order_details        margin calc          rank/window
dim_product
dim_product_subcat
dim_product_category
dim_territory
```

## 📋 Kolom Target: `fact_sales_performance`

| Kolom | Tipe | Keterangan |
|-------|------|------------|
| `product_id` | INT | ID produk |
| `product_name` | STRING | Nama produk |
| `category` | STRING | Kategori produk |
| `subcategory` | STRING | Subkategori produk |
| `territory_id` | INT | ID wilayah |
| `territory_name` | STRING | Nama wilayah |
| `country_code` | STRING | Kode negara |
| `order_year` | INT | Tahun order |
| `total_revenue` | DECIMAL | Total revenue (sum linetotal) |
| `total_qty` | INT | Total unit terjual |
| `avg_unit_price` | DECIMAL | Rata-rata harga jual |
| `list_price` | DECIMAL | Harga list produk |
| `standard_cost` | DECIMAL | Biaya standar produk |
| `margin` | DECIMAL | list_price - standard_cost |
| `margin_pct` | DECIMAL | margin / list_price * 100 |
| `load_timestamp` | TIMESTAMP | Waktu load |


## 0. Setup SparkSession

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *

spark = SparkSession.builder \
    .master('local[*]') \
    .appName('CaseA_SalesPerformance') \
    .config('hive.metastore.uris', 'thrift://hive-metastore:9083') \
    .config('spark.sql.warehouse.dir', 'hdfs://namenode:9000/user/hive/warehouse') \
    .enableHiveSupport() \
    .getOrCreate()

print(f'Spark {spark.version} ready')
print('Spark UI → http://localhost:4040')

# Cek database & tabel tersedia
spark.sql('SHOW DATABASES').show()
spark.sql('SHOW TABLES IN adventureworks').show(20)

## 1. Load Data dari Hive External Tables

Baca semua tabel yang dibutuhkan dari Hive menggunakan `spark.table()`.

Tabel ini sudah ada di Hive karena sudah dibuat di Day 40 (notebook `03_hdfs_to_hive.ipynb`).

In [ ]:
# Load semua tabel dari Hive External Tables

# Tabel fakta
df_orders  = spark.table('adventureworks.fact_sales_orders')    # header order
df_detail  = spark.table('adventureworks.fact_order_details')   # detail item per order

# Tabel dimensi
df_product  = spark.table('adventureworks.dim_product')          # info produk
df_prod_sub = spark.table('adventureworks.dim_product_subcat')   # subkategori produk
df_prod_cat = spark.table('adventureworks.dim_product_category') # kategori produk
df_territory = spark.table('adventureworks.dim_territory')       # wilayah penjualan

# Verifikasi jumlah baris
print('Row counts:')
for name, df in [
    ('fact_sales_orders',   df_orders),
    ('fact_order_details',  df_detail),
    ('dim_product',         df_product),
    ('dim_product_subcat',  df_prod_sub),
    ('dim_product_category',df_prod_cat),
    ('dim_territory',       df_territory),
]:
    print(f'  {name}: {df.count():,}')

## 2. Eksplorasi Data

In [ ]:
# Schema fact_sales_orders
print('=== Schema fact_sales_orders ===')
df_orders.printSchema()
df_orders.show(5)

In [ ]:
# Schema fact_order_details
print('=== Schema fact_order_details ===')
df_detail.printSchema()
df_detail.show(5)

In [ ]:
# Distribusi order per tahun
print('=== Distribusi order per tahun ===')
df_orders \
    .groupBy('order_year') \
    .agg(
        F.count('salesorderid').alias('total_orders'),
        F.round(F.sum('totaldue'), 2).alias('total_revenue')
    ) \
    .orderBy('order_year') \
    .show()

# Preview dimensi
print('=== dim_product sample ===')
df_product.select('productid', 'name', 'listprice', 'standardcost', 'productsubcategoryid').show(5)

print('=== dim_territory ===')
df_territory.show()

## 3. Extract & Enrich — Join antar DataFrame

Gabungkan semua tabel menjadi satu DataFrame yang kaya informasi.

Pipeline join:
```
fact_order_details
    JOIN fact_sales_orders   ON salesorderid
    JOIN dim_product         ON productid
    JOIN dim_product_subcat  ON productsubcategoryid
    JOIN dim_product_cat     ON productcategoryid
    JOIN dim_territory       ON territoryid   ← broadcast() karena tabel kecil
```

In [ ]:
# Siapkan kolom yang diperlukan dari tiap tabel (select dulu agar lebih efisien)

orders_slim = df_orders.select(
    'salesorderid', 'orderdate', 'territoryid', 'order_year', 'order_month'
)

product_slim = df_product.select(
    'productid',
    F.col('name').alias('product_name'),
    'listprice',
    'standardcost',
    'productsubcategoryid'
)

subcat_slim = df_prod_sub.select(
    F.col('productsubcategoryid').alias('sub_id'),
    F.col('name').alias('subcategory'),
    F.col('productcategoryid').alias('cat_id')
)

cat_slim = df_prod_cat.select(
    F.col('productcategoryid').alias('cat_id2'),
    F.col('name').alias('category')
)

territory_slim = df_territory.select(
    'territoryid',
    F.col('name').alias('territory_name'),
    F.col('countryregioncode').alias('country_code')
)

print('Kolom select selesai')

In [ ]:
# Join semua tabel menjadi satu DataFrame yang enriched
df_enriched = df_detail \
    .join(orders_slim,   on='salesorderid',           how='inner') \
    .join(product_slim,  on='productid',               how='inner') \
    .join(subcat_slim,   on=df_detail['productid'] == df_product['productid'],  how='left') \
    .join(subcat_slim.withColumnRenamed('sub_id', 'productsubcategoryid'),
          on='productsubcategoryid', how='left') \
    .join(cat_slim, on=F.col('cat_id') == F.col('cat_id2'), how='left') \
    .join(F.broadcast(territory_slim), on='territoryid', how='left')

# Cara yang lebih bersih (hindari ambiguitas kolom):
# Build step by step
df_enriched = df_detail.alias('od') \
    .join(orders_slim.alias('oh'),
          F.col('od.salesorderid') == F.col('oh.salesorderid'), 'inner') \
    .join(product_slim.alias('p'),
          F.col('od.productid') == F.col('p.productid'), 'inner') \
    .join(subcat_slim.alias('ps'),
          F.col('p.productsubcategoryid') == F.col('ps.sub_id'), 'left') \
    .join(cat_slim.alias('pc'),
          F.col('ps.cat_id') == F.col('pc.cat_id2'), 'left') \
    .join(F.broadcast(territory_slim).alias('t'),
          F.col('oh.territoryid') == F.col('t.territoryid'), 'left') \
    .select(
        F.col('od.salesorderid'),
        F.col('od.productid'),
        F.col('od.orderqty'),
        F.col('od.unitprice'),
        F.col('od.linetotal'),
        F.col('oh.territoryid'),
        F.col('oh.order_year'),
        F.col('oh.order_month'),
        F.col('p.product_name'),
        F.col('p.listprice'),
        F.col('p.standardcost'),
        F.col('ps.subcategory'),
        F.col('pc.category'),
        F.col('t.territory_name'),
        F.col('t.country_code')
    )

# Tambahkan kolom kalkulasi margin
df_enriched = df_enriched \
    .withColumn('margin',
        F.round(F.col('listprice') - F.col('standardcost'), 2)) \
    .withColumn('margin_pct',
        F.round(
            F.when(F.col('listprice') > 0,
                   (F.col('listprice') - F.col('standardcost')) / F.col('listprice') * 100
            ).otherwise(0),
        2))

print(f'Total rows setelah join: {df_enriched.count():,}')
df_enriched.printSchema()
df_enriched.show(5, truncate=False)

## 4. Transform — Agregasi Sales Performance

Buat agregasi per `(product, territory, year)` untuk menghasilkan tabel fact.

In [ ]:
# Agregasi per (product_id, territory_id, order_year)
df_perf = df_enriched \
    .groupBy(
        'productid',
        'product_name',
        'category',
        'subcategory',
        'territoryid',
        'territory_name',
        'country_code',
        'order_year'
    ) \
    .agg(
        F.round(F.sum('linetotal'), 2).alias('total_revenue'),
        F.sum('orderqty').alias('total_qty'),
        F.round(F.avg('unitprice'), 2).alias('avg_unit_price'),
        F.round(F.first('listprice'), 2).alias('list_price'),
        F.round(F.first('standardcost'), 2).alias('standard_cost'),
        F.round(F.first('margin'), 2).alias('margin'),
        F.round(F.first('margin_pct'), 2).alias('margin_pct')
    ) \
    .withColumnRenamed('productid', 'product_id') \
    .withColumnRenamed('territoryid', 'territory_id')

print(f'Total rows fact_sales_performance: {df_perf.count():,}')
df_perf.orderBy(F.desc('total_revenue')).show(10, truncate=False)

## 5. Window Functions — Ranking & Analisis Tren

### 5.1 Ranking Produk per Territory per Year

In [ ]:
# Window spec: ranking produk dalam tiap (order_year, territory_name)
win_spec = Window \
    .partitionBy('order_year', 'territory_name') \
    .orderBy(F.desc('total_revenue'))

df_ranked = df_perf \
    .withColumn('revenue_rank',   F.rank().over(win_spec)) \
    .withColumn('revenue_dense',  F.dense_rank().over(win_spec)) \
    .withColumn('territory_rank', F.row_number().over(win_spec))

# Top 3 produk per territory per year
print('=== Top 3 Produk per Territory per Year ===')
df_ranked \
    .filter(F.col('revenue_rank') <= 3) \
    .orderBy('order_year', 'territory_name', 'revenue_rank') \
    .select(
        'order_year', 'territory_name', 'revenue_rank',
        'product_name', 'category',
        'total_revenue', 'total_qty'
    ) \
    .show(30, truncate=False)

### 5.2 Cumulative Revenue & Month-over-Month Growth

In [ ]:
# Agregasi monthly revenue dari fact_sales_orders
df_monthly = df_orders \
    .groupBy('order_year', 'order_month') \
    .agg(
        F.count('salesorderid').alias('total_orders'),
        F.round(F.sum('totaldue'), 2).alias('monthly_revenue')
    ) \
    .orderBy('order_year', 'order_month')

# Window untuk cumulative revenue (running total dalam 1 tahun)
win_cum = Window \
    .partitionBy('order_year') \
    .orderBy('order_month') \
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)

# Window untuk lag (perbandingan bulan sebelumnya)
win_lag = Window \
    .partitionBy('order_year') \
    .orderBy('order_month')

df_monthly = df_monthly \
    .withColumn('cumulative_revenue',
        F.round(F.sum('monthly_revenue').over(win_cum), 2)) \
    .withColumn('prev_month_revenue',
        F.lag('monthly_revenue', 1).over(win_lag)) \
    .withColumn('mom_growth_pct',
        F.round(
            F.when(
                F.col('prev_month_revenue').isNotNull() & (F.col('prev_month_revenue') > 0),
                (F.col('monthly_revenue') - F.col('prev_month_revenue'))
                / F.col('prev_month_revenue') * 100
            ).otherwise(None),
        2))

print('=== Monthly Revenue 2013-2014 dengan Cumulative & MoM Growth ===')
df_monthly \
    .filter(F.col('order_year') >= 2013) \
    .orderBy('order_year', 'order_month') \
    .show(30)

## 6. Load — Simpan ke Hive Curated Layer

In [ ]:
# Buat database curated jika belum ada
spark.sql('CREATE DATABASE IF NOT EXISTS adventureworks_curated')

# Tambahkan load_timestamp
df_final = df_perf.withColumn('load_timestamp', F.current_timestamp())

# Simpan ke Hive sebagai managed table, partisi per order_year
df_final.write \
    .mode('overwrite') \
    .partitionBy('order_year') \
    .saveAsTable('adventureworks_curated.fact_sales_performance')

print('=== Verifikasi tabel tersimpan ===')
df_verify = spark.table('adventureworks_curated.fact_sales_performance')
print(f'Total rows: {df_verify.count():,}')
print(f'Partisi   : {df_verify.select("order_year").distinct().orderBy("order_year").collect()}')
df_verify.show(5, truncate=False)

## 7. Analytic Queries

Query dari tabel `adventureworks_curated.fact_sales_performance` yang sudah tersimpan.

### Query 1 (WAJIB): Top 10 Produk per Revenue Keseluruhan

In [ ]:
# Jawab BQ 1: Produk apa yang paling laris?
print('=== Query 1: Top 10 Produk by Revenue (semua tahun & territory) ===')

spark.sql("""
    SELECT
        product_name,
        category,
        subcategory,
        SUM(total_revenue)                      AS total_revenue,
        SUM(total_qty)                          AS total_qty,
        ROUND(AVG(avg_unit_price), 2)           AS avg_unit_price,
        ROUND(AVG(margin_pct), 2)               AS avg_margin_pct
    FROM adventureworks_curated.fact_sales_performance
    GROUP BY product_name, category, subcategory
    ORDER BY total_revenue DESC
    LIMIT 10
""").show(truncate=False)

### Query 2 (WAJIB): Revenue per Territory per Year + YoY Growth

In [ ]:
# Jawab BQ 2: Bagaimana tren revenue per tahun?
print('=== Query 2: Revenue per Territory per Year + YoY Growth ===')

# Agregasi revenue per territory per year
df_terr_year = spark.sql("""
    SELECT
        territory_name,
        country_code,
        order_year,
        ROUND(SUM(total_revenue), 2) AS yearly_revenue,
        SUM(total_qty)               AS yearly_qty
    FROM adventureworks_curated.fact_sales_performance
    GROUP BY territory_name, country_code, order_year
    ORDER BY territory_name, order_year
""")

# Tambahkan YoY growth menggunakan lag()
win_yoy = Window.partitionBy('territory_name').orderBy('order_year')

df_terr_yoy = df_terr_year \
    .withColumn('prev_year_revenue', F.lag('yearly_revenue', 1).over(win_yoy)) \
    .withColumn('yoy_growth_pct',
        F.round(
            F.when(
                F.col('prev_year_revenue').isNotNull() & (F.col('prev_year_revenue') > 0),
                (F.col('yearly_revenue') - F.col('prev_year_revenue'))
                / F.col('prev_year_revenue') * 100
            ).otherwise(None),
        2)) \
    .drop('prev_year_revenue')

df_terr_yoy.orderBy('territory_name', 'order_year').show(30, truncate=False)

### Query 3 (WAJIB): Produk dengan Margin Tertinggi

In [ ]:
# Jawab BQ 3: Produk mana yang margin-nya paling tinggi?
print('=== Query 3: Top 15 Produk by Margin % (revenue > 100,000) ===')

spark.sql("""
    SELECT
        product_name,
        category,
        subcategory,
        ROUND(AVG(list_price), 2)       AS list_price,
        ROUND(AVG(standard_cost), 2)    AS standard_cost,
        ROUND(AVG(margin), 2)           AS margin,
        ROUND(AVG(margin_pct), 2)       AS margin_pct,
        ROUND(SUM(total_revenue), 2)    AS total_revenue
    FROM adventureworks_curated.fact_sales_performance
    GROUP BY product_name, category, subcategory
    HAVING SUM(total_revenue) > 100000
    ORDER BY margin_pct DESC
    LIMIT 15
""").show(truncate=False)

### Query 4 (BONUS): Category Revenue Share per Year

In [ ]:
# Hitung % kontribusi tiap category terhadap total revenue per tahun
print('=== Query 4 (Bonus): Category Revenue Share per Year ===')

df_cat_year = spark.sql("""
    SELECT
        order_year,
        category,
        ROUND(SUM(total_revenue), 2) AS cat_revenue
    FROM adventureworks_curated.fact_sales_performance
    GROUP BY order_year, category
""")

# Window: sum total revenue per tahun untuk hitung persentase
win_year_total = Window.partitionBy('order_year')

df_cat_share = df_cat_year \
    .withColumn('yearly_total',
        F.sum('cat_revenue').over(win_year_total)) \
    .withColumn('revenue_share_pct',
        F.round(F.col('cat_revenue') / F.col('yearly_total') * 100, 2)) \
    .orderBy('order_year', F.desc('cat_revenue'))

df_cat_share.show(30)

### Query 5 (BONUS): Top 5 Produk per Territory per Year via Spark SQL

In [ ]:
# Jawab BQ 4: Top 5 produk per territory per year menggunakan pure SQL
print('=== Query 5 (Bonus): Top 5 Produk per Territory per Year ===')

spark.sql("""
    SELECT *
    FROM (
        SELECT
            order_year,
            territory_name,
            product_name,
            category,
            total_revenue,
            total_qty,
            margin_pct,
            RANK() OVER (
                PARTITION BY order_year, territory_name
                ORDER BY total_revenue DESC
            ) AS revenue_rank
        FROM adventureworks_curated.fact_sales_performance
    ) ranked
    WHERE revenue_rank <= 5
    ORDER BY order_year, territory_name, revenue_rank
""").show(50, truncate=False)

## 8. Summary & Cleanup

In [ ]:
# Tampilkan semua tabel yang sudah dibuat di curated layer
print('=== Tabel di adventureworks_curated ===')
spark.sql('SHOW TABLES IN adventureworks_curated').show()

# Final schema check
print('\n=== Schema fact_sales_performance ===')
spark.table('adventureworks_curated.fact_sales_performance').printSchema()

# Cleanup
spark.stop()
print('\nDone! Pipeline selesai.')
print('Cek tabel adventureworks_curated.fact_sales_performance di DBeaver/HiveServer2.')